# 07 · HITL：把规则服务失败交给人，再用同一 thread 恢复

上一节已经得到 status=need_human 和 TimeoutError。本节不继续自动重试，而是在独立节点调用 interrupt，把错误、订舱 ID 和可选动作交给人工审核。

恢复必须使用同一个 thread_id。Command(resume=...) 不是从某一代码行原地继续；包含 interrupt 的节点会重新执行，因此 interrupt 前不能放不可重复副作用。


In [ ]:
from __future__ import annotations

import operator
from collections import defaultdict
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class HitlState(TypedDict, total=False):
    booking_id: str
    status: str
    error_type: str
    tool_error: str
    human_decision: str
    manual_evidence: str
    final_status: str
    trace: Annotated[list[str], operator.add]


SERVICE_CALLS = defaultdict(int)


def failing_rule_service(booking_id: str) -> None:
    SERVICE_CALLS[booking_id] += 1
    raise TimeoutError(f"RULE_PRECHECK_TIMEOUT: booking_id={booking_id}")


def call_rule_service(state: HitlState) -> dict:
    try:
        failing_rule_service(state["booking_id"])
    except TimeoutError as exc:
        return {
            "status": "need_human",
            "error_type": type(exc).__name__,
            "tool_error": str(exc),
            "trace": ["rule_service:timeout", "status:need_human"],
        }
    raise AssertionError("课程故障服务应稳定超时")


# TODO(training): 完成规则服务失败后的 HITL 中断与恢复流程。
# 1. 实现 request_human_review：调用 interrupt(...) 暂停并返回人工决定与证据。
# 2. 实现 after_review：把人工决定路由到 accept 或 reject。
# 3. 实现 finalize_manual 与 close_rejected 两个收尾节点。
# 4. 创建 builder，注册上述节点及 call_rule_service，并连接条件边与 END。
# 完成后应得到 builder，供下方 builder.compile(checkpointer=...) 使用。


app = builder.compile(checkpointer=InMemorySaver())


## 1. accept：人工补证后恢复


In [ ]:
accept_config = {"configurable": {"thread_id": "rule-hitl-accept"}}
accept_input = {"booking_id": "BK-HITL-ACCEPT", "trace": []}

paused = app.invoke(accept_input, accept_config)
print("interrupt payload =", paused["__interrupt__"][0].value)
assert paused["__interrupt__"][0].value["error_type"] == "TimeoutError"
assert SERVICE_CALLS["BK-HITL-ACCEPT"] == 1


In [ ]:
# 审批通过
accepted = app.invoke(Command(resume={
    "decision": "accept_manual_evidence",
    "manual_evidence": "人工已从受控系统核对；演示证据编号 MANUAL-001",
}), accept_config)

print("accepted =", accepted)
assert accepted["final_status"] == "completed_by_human"
assert SERVICE_CALLS["BK-HITL-ACCEPT"] == 1, "resume 不应重跑上游故障工具节点"

## 2. reject：另一条 thread 明确停止


In [ ]:
reject_config = {"configurable": {"thread_id": "rule-hitl-reject"}}
rejected_pause = app.invoke({"booking_id": "BK-HITL-REJECT", "trace": []}, reject_config)

# 驳回
rejected = app.invoke(Command(resume={"decision": "reject"}), reject_config)

print("rejected =", rejected)
assert rejected["final_status"] == "stopped_by_human"
assert rejected["status"] == "rejected"
assert SERVICE_CALLS["BK-HITL-REJECT"] == 1
print("07 HITL accept/reject resume paths ok")


R## Router + HITL · 30 分钟双人联合练习

使用自己的 AI Coding 工具，把 06-router 的分流与本 Notebook 的 HITL 接成一条完整路径

联合通关证据：Router 路径可见；第一次 invoke 返回 __interrupt__；accept 得到 completed_by_human；reject 得到 stopped_by_human；失败没有伪装成功，也没有重复调用上游服务。


<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 第一次 invoke 得到 `__interrupt__`，并核对 payload 字段。
- [ ] 使用同一 `thread_id` 分别完成 accept 与 reject 路径。
- [ ] 证明 resume 后上游故障服务没有重复调用。
- [ ] 解释为什么 `interrupt()` 前不能放不可重复副作用。

**交付证据：**accept/reject 结果、服务调用计数、pause/resume event 类型。